# Whisper large-v3 — real qo'ng'iroqlarga moslash (round 3)Bu notebook oldingilaridan **boshqacha maqsad** bilan ishlaydi.| | Round 1–2 | **Round 3 (shu yerda)** ||---|---|---|| Maqsad | o'zbek TILINI o'rgatish | **telefon QO'NG'IROG'IGA moslash** || Ma'lumot | ochiq datasetlar (toza, mikrofon) | **o'z qo'ng'iroqlaringiz** + podkast || Boshlang'ich model | `openai/whisper-large-v3` | **round-2 modelingiz** || Eval | aralash ochiq ma'lumot | **faqat real qo'ng'iroqlar** |Sabab: round 2 eval WER'ni 34.01% → 27.82% ga tushirdi, lekin real qo'ng'iroqdasezilarli yaxshilanish bermadi. Chunki eval to'plami toza o'qilgan nutqdan iborat,qo'ng'iroq esa 8 kHz, siqilgan, shovqinli va erkin suhbat. Bu — **domen farqi**,til bilimi emas. Round 3 aynan shu farqni yopadi:1. **O'z qo'ng'iroqlaringiz** — 1022 namuna, 6.5 soat, Muxlisa transkripti bilan2. **Telefon augmentatsiyasi** — podkast audiosi 8 kHz'ga tushirilib, μ-law   kodek va shovqin qo'shiladi, ya'ni toza audio telefon audiosiga o'xshatiladi3. **Eval faqat qo'ng'iroqlarda** — o'lchov endi bizga kerak bo'lgan narsani o'lchaydi### ⚠️ Compute unit budjeti — 40 birlik| Bosqich | Taxminiy ||---|---|| O'rnatish + ma'lumot yuklash | ~4 birlik || Trening 5000 qadam (A100, batch 8) | ~28 birlik || Merge + WER o'lchash | ~3 birlik || **Jami** | **~35 birlik** |Pastdagi **budjet hisoblagichi** har 500 qadamda haqiqiy sarfni ko'rsatadi.Agar bashorat 40 dan oshsa, `MAX_STEPS` ni kamaytiring — checkpoint'lar Drive'dasaqlangani uchun hech narsa yo'qolmaydi.**GPU tanlash:** Runtime → Change runtime type → **A100**, High-RAM **shart emas**(audio diskdan o'qiladi, xotirada saqlanmaydi).

## 0. GPU va budjet

In [ ]:
import subprocess, torch, timeprint(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],                     capture_output=True, text=True).stdout.strip())GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YO'Q"# Colab Pro compute unit narxi (soatiga). Rasmiy jadval o'zgarishi mumkin —# bu faqat taxminiy hisob uchun.UNIT_RATE = {"A100": 11.77, "L4": 4.82, "V100": 4.91, "T4": 1.96}RATE = next((v for k, v in UNIT_RATE.items() if k in GPU), 11.77)print(f"GPU            : {GPU}")print(f"Birlik/soat    : ~{RATE}")if "A100" not in GPU:    print("\n⚠️  A100 emas. Boshqa GPU'da 5000 qadam ancha uzoq davom etadi.")    print("    Runtime → Change runtime type → A100 ni tanlang.")

## 1. Kutubxonalar

In [ ]:
!pip install -q -U transformers datasets accelerate peft jiwer soundfile "torchao>=0.16.0"# torchao pinni olib tashlamang: Colab'dagi eski versiya peft ichida ImportError beradi.print("✅ o'rnatildi")

## 2. Qo'ng'iroq dataseti (Google Drive)Avval **lokal kompyuterda** quyidagini bajaring:```bashcd ~/code/whisper-uzbek-asrpip install numpy soundfilepython scripts/prepare_calls_for_colab.py```Natijada `data/calls-colab/calls-colab.tar` (~286 MB) hosil bo'ladi —shu faylni Google Drive'ning **ildiziga (MyDrive)** yuklang.Skript nima qiladi: Muxlisa'ga audio 55 soniyalik bo'laklarda yuborilgan, Whisperesa 30 soniyadan uzunini qabul qilmaydi. Shuning uchun uzun namunalar **jimlikjoyidan** kesiladi va matn shu nuqtaga eng yaqin **gap chegarasidan** bo'linadi.Ishonch bo'lmasa namuna butunlay chetlatiladi — noto'g'ri moslashtirilgan namunamodelni buzadi. Shu yo'l bilan 238 namuna (1.12 soat) → **1022 namuna (6.51 soat)**.

In [ ]:
import os, tarfile, pandas as pdfrom google.colab import drivedrive.mount('/content/drive')TAR      = '/content/drive/MyDrive/calls-colab.tar'CALLS_DIR = '/content/calls'          # lokal SSD — Drive'dan o'qish sekinOUTPUT_DIR = '/content/drive/MyDrive/whisper-uz-calls'   # checkpointlar Drive'da!assert os.path.exists(TAR), f"{TAR} topilmadi — tar faylni MyDrive ildiziga yuklang"if not os.path.exists(f'{CALLS_DIR}/train.csv'):    os.makedirs(CALLS_DIR, exist_ok=True)    with tarfile.open(TAR) as t:        t.extractall(CALLS_DIR)    print("✅ ochildi")train_df = pd.read_csv(f'{CALLS_DIR}/train.csv')eval_df  = pd.read_csv(f'{CALLS_DIR}/eval.csv')for df in (train_df, eval_df):    df['audio'] = df['path'].apply(lambda p: f'{CALLS_DIR}/{p}')print(f"Qo'ng'iroq train : {len(train_df)} namuna")print(f"Qo'ng'iroq eval  : {len(eval_df)} namuna")print(f"Checkpoint       : {OUTPUT_DIR}")print(f"\nMisol: {train_df.sentence.iloc[0][:120]}...")

## 3. Podkast datasetiToshkent shevasidagi YouTube podkastlari — **tabiiy suhbat** nutqi, ya'niqo'ng'iroqqa eng yaqin ochiq manba. O'qib yozdirilgan datasetlar (Common Voice,UzbekVoice) bu yerda **atayin ishlatilmaydi**: ular juda toza va mikrofongayaqin, ular round 2 da allaqachon ishlatilgan.Podkast audiosi treningda **telefon augmentatsiyasidan** o'tadi (4-bo'limga qarang).

In [ ]:
from datasets import load_dataset, Dataset, Audio, concatenate_datasetsSR = 16000POD_ID = "BoburAmirov/podcasts_tashkent_dialect_youtube_uzbek_speech_dataset"pod = load_dataset(POD_ID, split="train")print("Ustunlar:", pod.column_names)# Audio ustunini NOMI bo'yicha emas, TURI bo'yicha topamiz — ba'zi datasetlarda# u "audio" emas, masalan "path" deb nomlangan.acol = next(n for n, f in pod.features.items() if isinstance(f, Audio))if acol != "audio":    pod = pod.rename_column(acol, "audio")    print(f"Audio ustuni: '{acol}' → 'audio'")tcol = next(c for c in ("text", "sentence", "transcript") if c in pod.column_names)pod = pod.map(lambda x: {"sentence": str(x[tcol]).strip()},              remove_columns=[c for c in pod.column_names if c != "audio"])pod = pod.filter(lambda x: len(x["sentence"]) >= 5)pod = pod.cast_column("audio", Audio(sampling_rate=SR))print(f"✅ Podkast: {len(pod)} namuna")

## 4. Aralashtirish — nisbat muhim941 qo'ng'iroq namunasini 14 500 podkast namunasi bilan shunchaki qo'shsak,qo'ng'iroqlar batch'larning atigi **6%** ini tashkil qiladi va model ulardandeyarli hech narsa o'rganmaydi.Shuning uchun qo'ng'iroqlarni **takrorlaymiz** (oversampling) — `CALL_RATIO`gacha. 0.35 — muvozanatli qiymat:* pastroq (0.2) → domen o'zgarishi kuchsiz, natija round 2 ga o'xshab qoladi* balandroq (0.6) → 941 namunani yodlab olish (overfitting) xavfi ortadiEval to'plami **takrorlanmaydi va augmentatsiya qilinmaydi** — u toza o'lchovbo'lib qolishi kerak. Eval qo'ng'iroqlari train'dagilardan **butun qo'ng'iroqbo'yicha** ajratilgan, ya'ni bitta suhbatning bo'laklari ikkala to'plamgabo'linib ketmagan.

In [ ]:
import mathfrom transformers import WhisperProcessorCALL_RATIO = 0.35MODEL_NAME = "Sunnat0091/whisper-large-v3-uz"   # round-2 modelingiz# Bazadan boshlamoqchi bo'lsangiz: "openai/whisper-large-v3" — lekin unda# 5000 qadam o'zbek tilini qaytadan o'rganishga ketadi, domenga emas.processor = WhisperProcessor.from_pretrained(MODEL_NAME, language="uzbek", task="transcribe")def to_ds(df, is_call):    d = Dataset.from_pandas(df[["audio", "sentence"]].reset_index(drop=True))    d = d.cast_column("audio", Audio(sampling_rate=SR))    return d.add_column("is_call", [is_call] * len(d))calls_tr = to_ds(train_df, 1)calls_ev = to_ds(eval_df, 1)pod_tr   = pod.add_column("is_call", [0] * len(pod))# Kerakli takrorlash soni: calls*R / (calls*R + pod) = CALL_RATIOR = max(1, round(CALL_RATIO * len(pod_tr) / ((1 - CALL_RATIO) * len(calls_tr))))train_ds = concatenate_datasets([calls_tr] * R + [pod_tr]).shuffle(seed=42)share = R * len(calls_tr) / len(train_ds)print(f"Qo'ng'iroq takrorlanishi : x{R}")print(f"Train hajmi              : {len(train_ds)} ({share:.0%} qo'ng'iroq)")print(f"Eval (faqat qo'ng'iroq)  : {len(calls_ev)}")# --- Tokenlashtirish (MODELNI YUKLASHDAN OLDIN) ---# Model GPU'da bo'lsa, .map() ko'p jarayonda qotib qoladi (CUDA + fork).MAX_LABEL = 448def tok(b):    ids = processor.tokenizer(b["sentence"]).input_ids    return {"labels": ids, "llen": len(ids)}train_ds = train_ds.map(tok, remove_columns=["sentence"], desc="Tokenlashtirish")calls_ev = calls_ev.map(tok, remove_columns=["sentence"], desc="Tokenlashtirish (eval)")before = len(train_ds)train_ds = train_ds.filter(lambda n: n <= MAX_LABEL, input_columns=["llen"]).remove_columns(["llen"])calls_ev = calls_ev.filter(lambda n: n <= MAX_LABEL, input_columns=["llen"]).remove_columns(["llen"])print(f"Uzun matnlar chiqarildi  : {before} → {len(train_ds)}")

## 5. Telefon augmentatsiyasiBu — round 3 ning **eng muhim qismi**. Podkast audiosi toza va keng polosali;qo'ng'iroq esa telefon tarmog'idan o'tgan. Farqni sun'iy ravishda yopamiz:| Qadam | Nima qiladi ||---|---|| 300–3400 Hz polosa | telefon liniyasi shu oraliqdan tashqarini o'tkazmaydi || 16 kHz → 8 kHz → 16 kHz | tarmoqning haqiqiy diskretlash chastotasi || μ-law (G.711) | telefoniyada ishlatiladigan kodekning aynan o'zi || Shovqin (SNR 12–30 dB) | liniya shovqini, fon ovozlari || Tasodifiy kuchaytirish | har xil mikrofon/apparat darajalari |Augmentatsiya **faqat podkastga** qo'llanadi (`is_call == 0`) — qo'ng'iroqlarallaqachon telefon audiosi, ularni ikkinchi marta buzish shart emas.Mel-spektrogramma ham shu yerda, **joyida** hisoblanadi. Oldindan hisoblabsaqlash large-v3 uchun namunasiga ~1.5 MB (128×3000 float32) — o'n minglabnamunada bu yuzlab GB va RAM tugashi degani.

In [ ]:
import numpy as np, torchfrom dataclasses import dataclassfrom typing import Any, Dict, Listfrom scipy.signal import butter, lfilter, resample_polyAUG_PROB = 0.75          # podkast namunasining necha foizi augmentatsiya qilinadi_B, _A = butter(4, [300 / (SR / 2), 3400 / (SR / 2)], btype="band")def mulaw(x, mu=255.0):    # G.711 μ-law: siqish → 8 bitga kvantlash → yozish. Telefon kodegi.    y = np.sign(x) * np.log1p(mu * np.abs(x)) / np.log1p(mu)    y = np.round(y * 127.0) / 127.0    return np.sign(y) * ((1 + mu) ** np.abs(y) - 1) / mudef phone_augment(x, rng):    x = lfilter(_B, _A, x).astype(np.float32)          # telefon polosasi    x = resample_poly(resample_poly(x, 1, 2), 2, 1)    # 16k → 8k → 16k    if rng.random() < 0.5:        x = mulaw(np.clip(x, -1, 1))    if rng.random() < 0.7:        snr = rng.uniform(12, 30)        p = np.mean(x ** 2) + 1e-12        x = x + rng.normal(0, np.sqrt(p / (10 ** (snr / 10))), len(x))    x = x * rng.uniform(0.6, 1.2)    return np.clip(x, -1, 1).astype(np.float32)@dataclassclass CallCollator:    processor: Any    decoder_start_token_id: int    augment: bool = True    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:        rng = np.random.default_rng()        audios = []        for f in features:            a = np.asarray(f["audio"]["array"], dtype=np.float32)            if self.augment and f["is_call"] == 0 and rng.random() < AUG_PROB:                a = phone_augment(a, rng)            audios.append(a)        batch = self.processor.feature_extractor(audios, sampling_rate=SR, return_tensors="pt")        labels_batch = self.processor.tokenizer.pad(            [{"input_ids": f["labels"]} for f in features], return_tensors="pt")        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():            labels = labels[:, 1:]        batch["labels"] = labels        return batchprint("✅ collator tayyor")

## 6. Model + LoRA

In [ ]:
from transformers import WhisperForConditionalGenerationfrom peft import LoraConfig, get_peft_modelmodel = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)# Modelni fp16'da yuklamang — PEFT bilan dtype nomuvofiqligi beradi.# Aralash aniqlik training_args dagi fp16=True (autocast) orqali boshqariladi.model = model.to("cuda")model.generation_config.language = "uzbek"model.generation_config.task = "transcribe"model.generation_config.forced_decoder_ids = Nonemodel.config.forced_decoder_ids = Nonemodel.config.use_cache = False# Muzlatilgan model + gradient checkpointing: kirish gradientni o'tkazishi uchunmodel.model.encoder.conv1.register_forward_hook(lambda m, i, o: o.requires_grad_(True))model = get_peft_model(model, LoraConfig(    r=32, lora_alpha=64, lora_dropout=0.05, bias="none",    target_modules=["q_proj", "v_proj"],))model.print_trainable_parameters()

## 7. Trening**Parametrlar nima uchun shunday:*** `batch_size=8` — A100'da 5000 qadam ~2.4 soat ≈ 28 birlik. Batch 16 bo'lsa  ~39 birlik bo'lib, 40 lik budjetga sig'maydi.* `learning_rate=1e-4` — LoRA adapterlari noldan boshlanadi, bu oldingi  round'larda ishlagan qiymat.* `save_steps=500` — checkpoint Drive'da. Birliklar tugab qolsa, oxirgi  checkpoint'dan davom ettirasiz (`resume_from_checkpoint=True`).* `load_best_model_at_end` + eval faqat qo'ng'iroqlarda — agar model 941 ta  qo'ng'iroq namunasini yodlab olsa (overfitting), eval loss ko'tariladi va  eng yaxshi checkpoint avtomatik tanlanadi.* `predict_with_generate` **ishlatilmaydi** — PEFT bilan mos kelmaydi  (`generate()` autocast'dan tashqarida ishga tushib dtype xatosi beradi).  WER trening tugagach 9-bo'limda alohida hisoblanadi.

In [ ]:
import timefrom transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallbackMAX_STEPS = 5000class BudgetCallback(TrainerCallback):    # Har 500 qadamda sarflangan va bashorat qilingan compute unit'ni chiqaradi.    def on_train_begin(self, args, state, control, **kw):        self.t0 = time.time()    def on_step_end(self, args, state, control, **kw):        s = state.global_step        if s % 500 or s == 0:            return        el = time.time() - self.t0        per = el / s        total_h = per * MAX_STEPS / 3600        print(f"  ⏱  {s}/{MAX_STEPS} | {per:.2f} s/qadam | sarflandi ~{el/3600*RATE:.1f} birlik"              f" | jami bashorat ~{total_h*RATE:.1f} birlik ({total_h:.1f} soat)")        if total_h * RATE > 38:            print("     ⚠️  Budjetdan oshmoqda! Treningni to'xtatib, oxirgi checkpoint'ni ishlating.")args = Seq2SeqTrainingArguments(    output_dir=OUTPUT_DIR,    per_device_train_batch_size=8,    per_device_eval_batch_size=8,    gradient_accumulation_steps=1,    gradient_checkpointing=True,    gradient_checkpointing_kwargs={"use_reentrant": False},    fp16=True,    learning_rate=1e-4,    warmup_steps=300,    max_steps=MAX_STEPS,    lr_scheduler_type="linear",    eval_strategy="steps",    eval_steps=500,    save_strategy="steps",    save_steps=500,    save_total_limit=3,    load_best_model_at_end=True,    metric_for_best_model="loss",    greater_is_better=False,    label_names=["labels"],    remove_unused_columns=False,    logging_steps=50,    report_to="none",    dataloader_num_workers=4,    push_to_hub=False,)trainer = Seq2SeqTrainer(    model=model, args=args,    train_dataset=train_ds,    eval_dataset=calls_ev,    data_collator=CallCollator(processor, model.config.decoder_start_token_id, augment=True),    processing_class=processor,    callbacks=[BudgetCallback()],)print(f"Qadam: {MAX_STEPS} | Batch: 8 | Train: {len(train_ds)} | Eval: {len(calls_ev)}")

In [ ]:
# Birinchi marta: resume=False. Birliklar tugab qolib qaytadan ulansangiz: True.RESUME = Falseresult = trainer.train(resume_from_checkpoint=RESUME or None)print(result)

## 8. SaqlashLoRA adapterlari asosiy modelga **birlashtiriladi** (`merge_and_unload`) va fp16sifatida saqlanadi — natija oddiy Whisper modeli bo'ladi, uni ishlatish uchun`peft` kerak emas. Aynan shu formatni RunPod handler'i kutadi.

In [ ]:
FINAL_DIR = OUTPUT_DIR + "-final"merged = trainer.model.merge_and_unload().half()merged.save_pretrained(FINAL_DIR)processor.save_pretrained(FINAL_DIR)print(f"✅ Saqlandi: {FINAL_DIR}")!du -sh {FINAL_DIR}

## 9. WER — real qo'ng'iroqlardaBu yerdagi raqam oldingi round'lardagi 27.82% bilan **taqqoslanmaydi**: u ochiqdatasetlarda, bu esa sizning qo'ng'iroqlaringizda o'lchanadi. Taqqoslash uchun**xuddi shu eval to'plamida** eski model ham o'lchanadi — faqat shu ikki raqamo'zaro ma'noga ega.

In [ ]:
import gc, jiwer, torchfrom transformers import pipelinenorm = jiwer.Compose([    jiwer.ToLowerCase(), jiwer.RemovePunctuation(),    jiwer.RemoveMultipleSpaces(), jiwer.Strip(),    jiwer.ReduceToListOfListOfWords(),])paths = eval_df["audio"].tolist()refs  = eval_df["sentence"].tolist()def measure(model_id, label):    pipe = pipeline("automatic-speech-recognition", model=model_id,                    torch_dtype=torch.float16, device="cuda", chunk_length_s=30)    hyps = [pipe(p, generate_kwargs={"language": "uzbek", "task": "transcribe"})["text"]            for p in paths]    wer = jiwer.wer(refs, hyps, truth_transform=norm, hypothesis_transform=norm)    print(f"{label:<28}: WER {wer*100:.2f}%")    del pipe; gc.collect(); torch.cuda.empty_cache()    return wer, hypsdel model, trainer, merged; gc.collect(); torch.cuda.empty_cache()new_wer, new_hyps = measure(FINAL_DIR,  "Round 3 (qo'ng'iroqqa moslangan)")old_wer, _        = measure(MODEL_NAME, "Round 2 (oldingi model)")print(f"\nFarq: {(old_wer-new_wer)*100:+.2f} foiz punkt "      f"({'yaxshilandi ✅' if new_wer < old_wer else 'yomonlashdi ❌'})")print("\n" + "─" * 70)for i in range(3):    print(f"\nHAQIQIY : {refs[i][:200]}")    print(f"MODEL   : {new_hyps[i][:200]}")

## 10. Hugging Face'ga yuklash → RunPodWER yaxshilangan bo'lsa, modelni HF'ga yuklaymiz va RunPod endpoint'iniyangilaymiz.**Token haqida:** tokenni to'g'ridan-to'g'ri katakka yozmang. Colab'ning chappanelidagi 🔑 (Secrets) bo'limiga `HF_TOKEN` nomi bilan qo'ying va "Notebookaccess" ni yoqing. Notebook'ni GitHub'ga saqlaganda token ichida qolib ketmaydi.

In [ ]:
from google.colab import userdatafrom huggingface_hub import HfApiREPO = "Sunnat0091/whisper-large-v3-uz-calls"   # yangi nom — eskisi tegilmasinapi = HfApi(token=userdata.get('HF_TOKEN'))api.create_repo(REPO, repo_type="model", exist_ok=True, private=True)api.upload_folder(folder_path=FINAL_DIR, repo_id=REPO, repo_type="model")print(f"✅ https://huggingface.co/{REPO}")print("\nKeyingi qadam — RunPod:")print("  1. Dockerfile'da HF_MODEL_ID ni yangi repoga o'zgartiring va push qiling")print("  2. RunPod → Endpoint → New Release (image qayta yig'iladi)")print("  3. python3 test_runpod.py <qo'ng'iroq.aac> bilan tekshiring")